# LeetCode 779 — K-th Symbol in Grammar bằng Python

Link đề: https://leetcode.com/problems/k-th-symbol-in-grammar/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation, Recursion (đệ quy), Math

Skeleton LeetCode:

```python
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        ...
```

Ghi chú: hàng thứ `n` dài `2^(n-1)` ký tự, với `n = 30` là hơn nửa tỷ: **không thể dựng cả hàng**. Bài thưởng cho ai nhìn ra **cấu trúc tự tương tự** của các hàng (nửa đầu = hàng trước, nửa sau = hàng trước đảo bit), từ đó có lời giải `O(n)` bằng đệ quy và thậm chí `O(1)` bằng **số bit 1 của `k - 1`** (dãy Thue-Morse).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bảng `n` hàng, hàng 1 là `0`, mỗi hàng sau thay `0 -> 01`, `1 -> 10` từng ký tự của hàng trước; trả ký tự thứ `k` (đánh số từ 1) của hàng `n`.
2. Nắm các công cụ Python cần thiết: `str.replace`/`join`, toán tử bit `<<`, `>>`, `&`, `^`, `int.bit_count()`, `bin(x).count("1")`, `divmod`, đệ quy và chuyển đệ quy thành vòng lặp.
3. Quan sát quy luật: **nửa đầu hàng `n` bằng hàng `n-1`; nửa sau là hàng `n-1` đảo bit**; và hai ký tự con của một ký tự cha là `(cha, đảo cha)`.
4. Đi qua 5 cách: dựng hàng trực tiếp, đệ quy theo nửa, leo ngược lên cha đếm số lần đảo, **parity của số bit 1** của `k - 1`, bảng DP Thue-Morse; đối chiếu mọi cách bằng oracle dựng hàng.
5. Tránh các lỗi kinh điển: lệch 1 giữa chỉ số 1 và chỉ số 0 (`k` hay `k - 1`), quên đảo bit ở nửa sau, dùng sai nửa (`2^(n-1)` thay vì `2^(n-2)`), dựng cả hàng gây hết thời gian/bộ nhớ.

## 2. Đề bài & ví dụ

Ta dựng một bảng gồm `n` hàng (đánh số từ 1). Hàng 1 viết số `0`. Ở mỗi hàng tiếp theo, nhìn hàng trước và thay mỗi `0` bằng `01`, mỗi `1` bằng `10`.

Ví dụ với `n = 3`: hàng 1 là `0`, hàng 2 là `01`, hàng 3 là `0110`.

Cho hai số nguyên `n` và `k`, trả về ký tự thứ `k` (**đánh số từ 1**) ở hàng thứ `n`.

**Ví dụ 1:** `n = 1, k = 1` -> `0` (hàng 1: `0`)

**Ví dụ 2:** `n = 2, k = 1` -> `0` (hàng 2: `01`, ký tự thứ 1 là `0`)

**Ví dụ 3:** `n = 2, k = 2` -> `1` (hàng 2: `01`, ký tự thứ 2 là `1`)

**Ràng buộc:**

- `1 <= n <= 30`
- `1 <= k <= 2^(n - 1)`

Lưu ý: độ dài hàng `n` là `2^(n-1)`, nên `k` có thể lớn tới `536.870.912`.

In [1]:
from typing import List
import random, time

examples = [
    (1, 1, 0),
    (2, 1, 0),
    (2, 2, 1),
]
for n, k, expected in examples:
    print(f"n={n} k={k} độ dài hàng n = {2 ** (n - 1)} kỳ vọng={expected}")

n=1 k=1 độ dài hàng n = 1 kỳ vọng=0
n=2 k=1 độ dài hàng n = 2 kỳ vọng=0
n=2 k=2 độ dài hàng n = 2 kỳ vọng=1


## 3. Kiến thức nền

### 3.1. Thay thế chuỗi: `replace` và `join`

Quy tắc `0 -> 01`, `1 -> 10` phải áp dụng **đồng thời** lên các ký tự của hàng cũ. Gọi `str.replace` lần lượt sẽ sai (sau khi đổi `0` thành `01`, các `1` mới sinh ra lại bị đổi). Cách đúng là duyệt từng ký tự và nối kết quả bằng `"".join`.

In [2]:
row = "0110"
wrong = row.replace("0", "01").replace("1", "10")      # SAI: '1' mới sinh ra bị đổi lần nữa
right = "".join("01" if c == "0" else "10" for c in row)
print("sai :", wrong)
print("đúng:", right)

sai : 0101010010
đúng: 01101001


### 3.2. Chỉ số 1 và chỉ số 0

Đề đánh số ký tự từ **1**, còn chuỗi Python đánh từ **0**: ký tự thứ `k` là `row[k - 1]`. Rất nhiều công thức ở bài này đẹp hơn khi dùng `i = k - 1` (chỉ số từ 0).

In [3]:
row = "01101001"
for k in (1, 2, 5, 8):
    print(f"ký tự thứ {k} (1-indexed) = row[{k - 1}] = {row[k - 1]}")

ký tự thứ 1 (1-indexed) = row[0] = 0
ký tự thứ 2 (1-indexed) = row[1] = 1
ký tự thứ 5 (1-indexed) = row[4] = 1
ký tự thứ 8 (1-indexed) = row[7] = 1


### 3.3. Toán tử bit

- `1 << m` là `2^m`; `x >> 1` là chia nguyên cho 2; `x & 1` là bit thấp nhất (tính chẵn/lẻ).
- `x ^ 1` đảo bit thấp nhất: `0 ^ 1 = 1`, `1 ^ 1 = 0`. Ở bài này `^ 1` chính là "đảo ký tự".
- `x.bit_count()` (Python 3.10+) đếm số bit 1; `bin(x).count("1")` cho kết quả như nhau và chạy trên mọi phiên bản.

In [4]:
for x in (0, 1, 2, 5, 6, 7, 12):
    print(f"x={x:2} nhị phân={bin(x):7} bit_count={x.bit_count()} bin().count('1')={bin(x).count('1')} parity={x.bit_count() & 1}")
print("1 << 4 =", 1 << 4, "| 13 >> 1 =", 13 >> 1, "| 13 & 1 =", 13 & 1, "| 0 ^ 1 =", 0 ^ 1, "| 1 ^ 1 =", 1 ^ 1)

x= 0 nhị phân=0b0     bit_count=0 bin().count('1')=0 parity=0
x= 1 nhị phân=0b1     bit_count=1 bin().count('1')=1 parity=1
x= 2 nhị phân=0b10    bit_count=1 bin().count('1')=1 parity=1
x= 5 nhị phân=0b101   bit_count=2 bin().count('1')=2 parity=0
x= 6 nhị phân=0b110   bit_count=2 bin().count('1')=2 parity=0
x= 7 nhị phân=0b111   bit_count=3 bin().count('1')=3 parity=1
x=12 nhị phân=0b1100  bit_count=2 bin().count('1')=2 parity=0
1 << 4 = 16 | 13 >> 1 = 6 | 13 & 1 = 1 | 0 ^ 1 = 1 | 1 ^ 1 = 0


### 3.4. Đệ quy và độ sâu

Đệ quy là hàm gọi lại chính nó với bài toán nhỏ hơn, cần **điều kiện dừng**. Ở bài này mỗi lần gọi giảm `n` đi 1, nên độ sâu tối đa `n <= 30`: rất an toàn (giới hạn mặc định của Python là khoảng 1000).

In [5]:
def depth_demo(n):
    if n == 0:
        return 0
    return 1 + depth_demo(n - 1)

print(depth_demo(30))

30


### 3.5. `divmod`

`divmod(a, b)` trả `(a // b, a % b)` cùng lúc. Sẽ dùng để chia `k` thành "thứ tự cặp" và "vị trí trong cặp".

In [6]:
for k in (1, 2, 3, 4, 7, 8):
    pair, pos = divmod(k - 1, 2)
    print(f"k={k}: chỉ số {k - 1} -> cặp số {pair}, vị trí trong cặp {pos}")

k=1: chỉ số 0 -> cặp số 0, vị trí trong cặp 0
k=2: chỉ số 1 -> cặp số 0, vị trí trong cặp 1
k=3: chỉ số 2 -> cặp số 1, vị trí trong cặp 0
k=4: chỉ số 3 -> cặp số 1, vị trí trong cặp 1
k=7: chỉ số 6 -> cặp số 3, vị trí trong cặp 0
k=8: chỉ số 7 -> cặp số 3, vị trí trong cặp 1


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Viết vài hàng đầu:

```
n = 1: 0
n = 2: 0 1
n = 3: 0 1 1 0
n = 4: 0 1 1 0 1 0 0 1
n = 5: 0 1 1 0 1 0 0 1 1 0 0 1 0 1 1 0
```

Hai quan sát:

1. **Nửa đầu của hàng `n` = hàng `n - 1`; nửa sau = hàng `n - 1` đảo bit** (0 thành 1, 1 thành 0). Ví dụ hàng 4 = `0110` + `1001`, và `1001` là `0110` đảo.
2. **Cha-con:** ký tự thứ `j` của hàng `n - 1` sinh ra hai ký tự thứ `2j - 1` và `2j` của hàng `n`: ký tự **lẻ** (`2j - 1`) **bằng** cha, ký tự **chẵn** (`2j`) **đảo** cha. Ví dụ hàng 3 ký tự thứ 3 (lẻ) là con của ký tự thứ 2 ở hàng 2 (`1`), nên bằng `1`; ký tự thứ 4 (chẵn) là `0`, đảo.

Quan sát 2 cho phép "leo ngược" từ `(n, k)` về `(1, 1)` (luôn là `0`), đếm số lần gặp vị trí chẵn (số lần đảo): chẵn lần đảo thì `0`, lẻ lần thì `1`.

In [7]:
def build_row(n: int) -> str:
    row = "0"
    for _ in range(n - 1):
        row = "".join("01" if c == "0" else "10" for c in row)
    return row

for n in range(1, 6):
    row = build_row(n)
    print(f"n={n}: {row}   (độ dài {len(row)})")

r4, r3 = build_row(4), build_row(3)
flip = lambda s: "".join("1" if c == "0" else "0" for c in s)
print("hàng 4 = hàng 3 + hàng 3 đảo ?", r4 == r3 + flip(r3))

n=1: 0   (độ dài 1)
n=2: 01   (độ dài 2)
n=3: 0110   (độ dài 4)
n=4: 01101001   (độ dài 8)
n=5: 0110100110010110   (độ dài 16)
hàng 4 = hàng 3 + hàng 3 đảo ? True


## 5. Cách 1 — Brute Force: dựng cả hàng

**Ý tưởng.** Làm đúng như đề: bắt đầu từ `"0"`, lặp `n - 1` lần thay `0 -> 01`, `1 -> 10`, rồi lấy `row[k - 1]`.

Đây là định nghĩa trực tiếp nên cũng là **oracle** (chắc chắn đúng) cho `n` nhỏ.

In [8]:
def kth_build(n: int, k: int) -> int:
    row = "0"
    for _ in range(n - 1):
        row = "".join("01" if c == "0" else "10" for c in row)
    return int(row[k - 1])

for n, k, expected in examples:
    got = kth_build(n, k)
    print(n, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

1 1 -> 0 OK
2 1 -> 0 OK
2 2 -> 1 OK


**Độ phức tạp và nút thắt.** Hàng `n` có `2^(n-1)` ký tự, tổng công việc `O(2^n)` thời gian và `O(2^n)` bộ nhớ. Với `n = 30`: hơn `5 * 10^8` ký tự (nhiều GB bộ nhớ trong Python). Hãy xem độ dài tăng ra sao và thời gian tăng gấp đôi mỗi hàng:

In [9]:
for n in (10, 14, 18, 20, 22):
    t0 = time.perf_counter()
    kth_build(n, 1)
    dt = (time.perf_counter() - t0) * 1000
    print(f"n={n:2}: độ dài hàng {2 ** (n - 1):>9,}  thời gian {dt:8.1f} ms")
print("n = 30 sẽ có", f"{2 ** 29:,}", "ký tự: không khả thi")

n=10: độ dài hàng       512  thời gian      0.0 ms
n=14: độ dài hàng     8,192  thời gian      0.4 ms
n=18: độ dài hàng   131,072  thời gian      7.0 ms
n=20: độ dài hàng   524,288  thời gian     25.6 ms
n=22: độ dài hàng 2,097,152  thời gian     91.9 ms
n = 30 sẽ có 536,870,912 ký tự: không khả thi


Nút thắt: ta chỉ cần **một** ký tự mà phải dựng **toàn bộ** hàng. Quy luật cấu trúc cho phép tính trực tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

Dùng hai quy luật ở mục 4:

- **Chia đôi (Cách 2):** hàng `n` có độ dài `2^(n-1)`, nửa đầu `2^(n-2)`. Nếu `k <= 2^(n-2)`: đáp án là ký tự thứ `k` của hàng `n - 1`. Ngược lại nó là **đảo** của ký tự thứ `k - 2^(n-2)` của hàng `n - 1`.
- **Cha-con (Cách 3):** ký tự thứ `k` ở hàng `n` là con của ký tự thứ `ceil(k / 2) = (k + 1) // 2` ở hàng `n - 1`; nếu `k` **chẵn** thì đảo cha, nếu lẻ thì bằng cha. Mỗi bước nửa `k`, tổng cộng `O(log k)` bước; đáp án **không phụ thuộc `n`**.
- **Bit 1 (Cách 4):** mỗi lần `k` chẵn ta đảo một lần. Viết `k - 1` ở nhị phân: bước "leo lên cha" chính là dịch phải một bit; `k` chẵn ⇔ `k - 1` lẻ ⇔ bit thấp nhất của `k - 1` là 1. Do đó số lần đảo = **số bit 1 của `k - 1`**, đáp án = `popcount(k - 1) mod 2` (dãy **Thue-Morse**).
- **DP bảng (Cách 5):** `t[i] = t[i // 2] ^ (i & 1)` cho `i = 0, 1, 2, ...` với `t[0] = 0` — cùng một công thức cha-con ở dạng chỉ số 0, điền bảng tới `k - 1` (tốn `O(k)`).

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy theo nửa: nửa đầu giữ, nửa sau đảo |
| 3 | Leo ngược theo cha: `k -> (k + 1) // 2`, đếm lần `k` chẵn |
| 4 | `popcount(k - 1) & 1` |
| 5 | Bảng `t[i] = t[i // 2] ^ (i & 1)` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy theo nửa

**Giải thích từng đoạn.**

- Điều kiện dừng: `n == 1` -> `0`.
- `half = 1 << (n - 2)` là độ dài **nửa** hàng `n` (bằng độ dài hàng `n - 1`).
- Nếu `k <= half`: `kthGrammar(n - 1, k)`.
- Ngược lại: `kthGrammar(n - 1, k - half) ^ 1` (đảo bit).

In [10]:
def kth_halves(n: int, k: int) -> int:
    if n == 1:
        return 0
    half = 1 << (n - 2)
    if k <= half:
        return kth_halves(n - 1, k)
    return kth_halves(n - 1, k - half) ^ 1

for n, k, expected in examples:
    assert kth_halves(n, k) == expected
print("đệ quy theo nửa OK")

đệ quy theo nửa OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        if n == 1:
            return 0
        if k <= (1 << (n - 2)):
            return self.kthGrammar(n - 1, k)
        return self.kthGrammar(n - 1, k - (1 << (n - 2))) ^ 1

sol_halves = Solution().kthGrammar
print([sol_halves(n, k) for n, k, _ in examples])

[0, 0, 1]


**Độ phức tạp (Cách 2):** thời gian `O(n)` (mỗi lần gọi giảm `n` đi 1), bộ nhớ `O(n)` ngăn xếp đệ quy.

---

### 7.2. Cách 3 — Leo ngược lên cha, đếm số lần đảo

**Ý tưởng khác.** Thay vì đi từ hàng `n` xuống, đi từ `(n, k)` **lên** `(1, 1)` bằng quan hệ cha-con: cha của vị trí `k` là `(k + 1) // 2`; con ở vị trí **chẵn** là đảo của cha. Mỗi bước lên nếu `k` chẵn thì ta đã đảo một lần.

**Giải thích từng đoạn.**

- `flips = 0`.
- Khi `k > 1`: nếu `k` chẵn thì `flips += 1`; rồi `k = (k + 1) // 2`.
- Khi `k = 1` (gốc, ký tự `0`): đáp án là `flips & 1` (đảo chẵn lần ra `0`, lẻ lần ra `1`).
- Biến `n` **không dùng** vì cấu trúc của `k` đã quyết định đường đi (miễn `k <= 2^(n-1)`).

In [12]:
def kth_parent(n: int, k: int) -> int:
    flips = 0
    while k > 1:
        if k % 2 == 0:
            flips += 1
        k = (k + 1) // 2
    return flips & 1

for n, k, expected in examples:
    assert kth_parent(n, k) == expected
print("leo cha OK")

leo cha OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        flips = 0
        while k > 1:
            if k % 2 == 0:
                flips += 1
            k = (k + 1) // 2
        return flips & 1

sol_parent = Solution().kthGrammar
print([sol_parent(n, k) for n, k, _ in examples])

[0, 0, 1]


**Độ phức tạp (Cách 3):** thời gian `O(log k)` (<= 30 vòng lặp), bộ nhớ `O(1)`.

---

### 7.3. Cách 4 — Parity của số bit 1 của `k - 1` (Thue-Morse)

**Ý tưởng.** Cách 3 đếm số `k` chẵn trên đường leo. Với `i = k - 1`: `k` chẵn ⇔ `i` lẻ ⇔ bit thấp nhất của `i` là 1; còn bước `k -> (k + 1) // 2` tương ứng `i -> i // 2` (dịch phải). Vậy tổng số lần đảo = **số bit 1 của `i = k - 1`**. Đáp án là `popcount(k - 1) mod 2`.

**Giải thích từng đoạn.** Chỉ một dòng: `(k - 1).bit_count() & 1`. Để chạy trên Python cũ có thể dùng `bin(k - 1).count("1") & 1`.

In [14]:
def kth_popcount(n: int, k: int) -> int:
    return (k - 1).bit_count() & 1

def kth_popcount_bin(n: int, k: int) -> int:
    return bin(k - 1).count("1") & 1

for n, k, expected in examples:
    assert kth_popcount(n, k) == kth_popcount_bin(n, k) == expected
print("popcount OK")

popcount OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        return (k - 1).bit_count() & 1

sol_popcount = Solution().kthGrammar
print([sol_popcount(n, k) for n, k, _ in examples])

[0, 0, 1]


**Độ phức tạp (Cách 4):** thời gian `O(1)` (đếm bit của một số 30 bit), bộ nhớ `O(1)`.

---

### 7.4. Cách 5 — Bảng DP Thue-Morse

**Ý tưởng khác.** Dùng quan hệ cha-con ở chỉ số 0: ký tự ở chỉ số `i` là con của chỉ số `i // 2` ở hàng trước; con ở vị trí lẻ (`i & 1 == 1`) thì **đảo** cha. Vậy `t[i] = t[i // 2] ^ (i & 1)`, `t[0] = 0`. Điền bảng tới `k - 1`.

**Giải thích từng đoạn.**

- `t = [0] * k`.
- `for i in range(1, k): t[i] = t[i >> 1] ^ (i & 1)`.
- Đáp án `t[k - 1]`.
- Cách này tốn `O(k)` nên **không** dùng cho `k` tới nửa tỷ trong Python; nó minh họa bảng DP và đối chiếu tốt cho `k` vừa phải.

In [16]:
def kth_table(n: int, k: int) -> int:
    t = [0] * k
    for i in range(1, k):
        t[i] = t[i >> 1] ^ (i & 1)
    return t[k - 1]

for n, k, expected in examples:
    assert kth_table(n, k) == expected
print("bảng DP OK")

bảng DP OK


**Code nộp LeetCode (Cách 5)** (chỉ phù hợp `k` vừa phải; cho `k` lớn sẽ vượt bộ nhớ):

In [17]:
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        t = [0] * k
        for i in range(1, k):
            t[i] = t[i >> 1] ^ (i & 1)
        return t[k - 1]

sol_table = Solution().kthGrammar
print([sol_table(n, k) for n, k, _ in examples])

[0, 0, 1]


**Độ phức tạp (Cách 5):** thời gian `O(k)`, bộ nhớ `O(k)`. Chậm hơn hẳn các cách trước khi `k` lớn; ở đề `k <= 2^29` sẽ hết bộ nhớ.

## 8. TRACE từng bước

### 8.1. Cách 2 cho `n = 4, k = 6`

Hàng 4 là `01101001`, ký tự thứ 6 là `0`.

| lời gọi | `half` | `k <= half`? | hành động |
|---|---|---|---|
| `f(4, 6)` | 4 | không (6 > 4) | `f(3, 2) ^ 1` |
| `f(3, 2)` | 2 | có | `f(2, 2)` |
| `f(2, 2)` | 1 | không (2 > 1) | `f(1, 1) ^ 1` |
| `f(1, 1)` | | | `0` |

Quay lui: `f(2,2) = 0 ^ 1 = 1`; `f(3,2) = 1`; `f(4,6) = 1 ^ 1 = 0`.

In [18]:
def trace_halves(n, k, depth=0):
    pad = "  " * depth
    if n == 1:
        print(f"{pad}f({n}, {k}) = 0 (hàng 1)")
        return 0
    half = 1 << (n - 2)
    if k <= half:
        print(f"{pad}f({n}, {k}): half={half}, k <= half -> f({n - 1}, {k})")
        r = trace_halves(n - 1, k, depth + 1)
        print(f"{pad}f({n}, {k}) = {r}")
        return r
    print(f"{pad}f({n}, {k}): half={half}, k > half -> f({n - 1}, {k - half}) ^ 1")
    r = trace_halves(n - 1, k - half, depth + 1) ^ 1
    print(f"{pad}f({n}, {k}) = {r}")
    return r

assert trace_halves(4, 6) == 0 == int(build_row(4)[5])

f(4, 6): half=4, k > half -> f(3, 2) ^ 1
  f(3, 2): half=2, k <= half -> f(2, 2)
    f(2, 2): half=1, k > half -> f(1, 1) ^ 1
      f(1, 1) = 0 (hàng 1)
    f(2, 2) = 1
  f(3, 2) = 1
f(4, 6) = 0


### 8.2. Cách 3 cho `n = 4, k = 6`: leo cha

| bước | `k` | `k` chẵn? | `flips` | `k` mới |
|---|---|---|---|---|
| 1 | 6 | có | 1 | 3 |
| 2 | 3 | không | 1 | 2 |
| 3 | 2 | có | 2 | 1 |
| 4 | 1 | dừng | | |

`flips = 2` (chẵn) -> `0`.

In [19]:
def trace_parent(n, k):
    flips = 0
    print(f"bắt đầu (n={n}, k={k})")
    while k > 1:
        even = k % 2 == 0
        if even:
            flips += 1
        parent = (k + 1) // 2
        print(f"  k={k:3} {'chẵn -> đảo' if even else 'lẻ  -> giữ'}; cha là vị trí {parent} của hàng trước; flips={flips}")
        k = parent
    print(f"  tới gốc k=1 (ký tự 0), flips={flips} -> kết quả {flips & 1}")
    return flips & 1

assert trace_parent(4, 6) == 0
print()
assert trace_parent(5, 11) == int(build_row(5)[10])

bắt đầu (n=4, k=6)
  k=  6 chẵn -> đảo; cha là vị trí 3 của hàng trước; flips=1
  k=  3 lẻ  -> giữ; cha là vị trí 2 của hàng trước; flips=1
  k=  2 chẵn -> đảo; cha là vị trí 1 của hàng trước; flips=2
  tới gốc k=1 (ký tự 0), flips=2 -> kết quả 0

bắt đầu (n=5, k=11)
  k= 11 lẻ  -> giữ; cha là vị trí 6 của hàng trước; flips=0
  k=  6 chẵn -> đảo; cha là vị trí 3 của hàng trước; flips=1
  k=  3 lẻ  -> giữ; cha là vị trí 2 của hàng trước; flips=1
  k=  2 chẵn -> đảo; cha là vị trí 1 của hàng trước; flips=2
  tới gốc k=1 (ký tự 0), flips=2 -> kết quả 0


### 8.3. Cách 4: số bit 1 của `k - 1`

Với `k = 6`: `k - 1 = 5 = 101` (hai bit 1) -> chẵn -> `0`. Với `k = 11`: `10 = 1010` (hai bit 1) -> `0`; kiểm tra hàng 5 `0110100110010110`: ký tự thứ 11 là `0` ✓. Bảng cho hàng 4: `k` từ 1 đến 8.

In [20]:
def trace_popcount(n):
    row = build_row(n)
    print(f"hàng {n}: {row}")
    print(f"{'k':>3} {'k-1':>4} {'nhị phân':>9} {'số bit 1':>9} {'parity':>7} {'ký tự thật':>11}")
    for k in range(1, 2 ** (n - 1) + 1):
        i = k - 1
        print(f"{k:>3} {i:>4} {i:>9b} {i.bit_count():>9} {i.bit_count() & 1:>7} {row[k - 1]:>11}")
        assert (i.bit_count() & 1) == int(row[k - 1])

trace_popcount(4)

hàng 4: 01101001
  k  k-1  nhị phân  số bit 1  parity  ký tự thật
  1    0         0         0       0           0
  2    1         1         1       1           1
  3    2        10         1       1           1
  4    3        11         2       0           0
  5    4       100         1       1           1
  6    5       101         2       0           0
  7    6       110         2       0           0
  8    7       111         3       1           1


### 8.4. Cách 5: bảng `t`

`t[i] = t[i // 2] ^ (i & 1)` cho `i = 0..7` tái tạo chính hàng 4:

In [21]:
t = [0] * 8
for i in range(1, 8):
    t[i] = t[i >> 1] ^ (i & 1)
    print(f"i={i}: t[{i}] = t[{i >> 1}] ^ {i & 1} = {t[i]}")
print("".join(map(str, t)), "== hàng 4:", build_row(4), "?", "".join(map(str, t)) == build_row(4))
assert "".join(map(str, t)) == build_row(4)

i=1: t[1] = t[0] ^ 1 = 1
i=2: t[2] = t[1] ^ 0 = 1
i=3: t[3] = t[1] ^ 1 = 0
i=4: t[4] = t[2] ^ 0 = 1
i=5: t[5] = t[2] ^ 1 = 0
i=6: t[6] = t[3] ^ 0 = 0
i=7: t[7] = t[3] ^ 1 = 1
01101001 == hàng 4: 01101001 ? True


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (đệ quy theo nửa) hoặc **Cách 4** (popcount): Cách 2 gắn sát với cấu trúc của đề và dễ giải thích; Cách 4 là một dòng `O(1)`. Dưới đây nộp Cách 4 và kiểm chứng; hai cách còn lại đã kiểm ở trên.

```python
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        return (k - 1).bit_count() & 1
```

Nếu muốn lời giải "đúng theo cấu trúc" hơn khi phỏng vấn, dùng bản đệ quy:

```python
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        if n == 1:
            return 0
        if k <= (1 << (n - 2)):
            return self.kthGrammar(n - 1, k)
        return self.kthGrammar(n - 1, k - (1 << (n - 2))) ^ 1
```

In [22]:
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        return (k - 1).bit_count() & 1

sol = Solution()
for n, k, expected in examples:
    got = sol.kthGrammar(n, k)
    print(n, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected
print("n = 30, k = 2^29 ->", sol.kthGrammar(30, 2 ** 29), "| n = 30, k = 1 ->", sol.kthGrammar(30, 1))

1 1 -> 0 OK
2 1 -> 0 OK
2 2 -> 1 OK
n = 30, k = 2^29 -> 1 | n = 30, k = 1 -> 0


## 10. Vì sao đúng

**(a) Nửa đầu bằng hàng trước, nửa sau là đảo.** Hàng `n` được tạo bằng cách thay mỗi ký tự `c` của hàng `n - 1` bằng cặp `(c, ~c)`. Ta chứng minh quy nạp rằng hàng `n` = hàng `n-1` + đảo(hàng `n-1`). Với `n = 2`: `01 = 0 + 1` ✓. Giả sử hàng `n - 1` = `A + ~A` (với `A` là hàng `n - 2`). Thay từng ký tự thành cặp: kết quả là `[các cặp của A] + [các cặp của ~A]`. Các cặp của `A` chính là hàng `n - 1`; mỗi cặp của `~A` là `(~c, c)` = đảo của `(c, ~c)`, nên các cặp của `~A` chính là đảo của hàng `n - 1`. Vậy hàng `n` = hàng `n-1` + đảo(hàng `n-1`) ✓. Cách 2 là hệ quả trực tiếp.

**(b) Cha-con (Cách 3).** Ký tự thứ `j` của hàng `n - 1` sinh cặp `(c_j, ~c_j)` ở vị trí `2j - 1` và `2j` của hàng `n`. Vị trí lẻ `2j - 1` bằng `c_j`, vị trí chẵn `2j` đảo `c_j`. Với `k` bất kỳ, cha là `j = ceil(k / 2) = (k + 1) // 2`, và `k` chẵn ⇔ đảo.

**(c) Popcount (Cách 4).** Đặt `i = k - 1`. Khi `k` chẵn thì `i` lẻ, bit thấp nhất của `i` là 1. `k' = (k + 1) // 2` nên `i' = k' - 1 = (k + 1) // 2 - 1 = (k - 1) // 2 = i // 2`: dịch phải một bit. Lặp đến `k = 1` (`i = 0`): số lần đảo bằng số bit 1 của `i`. Gốc là `0`, nên kết quả = parity của số bit 1.

**(d) Cách 5.** `t[i] = t[i // 2] ^ (i & 1)` chính là (b) viết theo chỉ số 0: cha của chỉ số `i` là `i // 2`; con ở chỉ số lẻ đảo cha.

**Kiểm chứng bằng máy:** so mọi cách với oracle dựng hàng trên **mọi** cặp `(n, k)` với `n <= 12`, và so các cách nhanh với nhau (cùng Cách 5 khi `k` vừa) trên `n` lớn tới 30.

In [23]:
solutions = {
    "build": kth_build,
    "halves": kth_halves,
    "parent": kth_parent,
    "popcount": kth_popcount,
    "popcount_bin": kth_popcount_bin,
    "table": kth_table,
    "sol_halves": sol_halves,
    "sol_parent": sol_parent,
    "sol_popcount": sol_popcount,
    "sol_table": sol_table,
    "final": Solution().kthGrammar,
}

checked = 0
for n in range(1, 13):
    row = build_row(n)
    for k in range(1, 2 ** (n - 1) + 1):
        ref = int(row[k - 1])
        for name, fn in solutions.items():
            if name == "build" and n > 8:
                continue                                  # dựng lại cho mỗi k là thừa; đã dùng row làm oracle
            assert fn(n, k) == ref, (name, n, k)
        checked += 1
print("mọi cặp (n, k) với n <= 12:", checked, "cặp x", len(solutions), "cài đặt khớp oracle")

mọi cặp (n, k) với n <= 12: 4095 cặp x 11 cài đặt khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Dựng hàng | `O(2^n)` | `O(2^n)` |
| 2. Đệ quy theo nửa | `O(n)` | `O(n)` ngăn xếp |
| 3. Leo cha | `O(log k)` | `O(1)` |
| 4. Popcount | `O(1)` (số 30 bit) | `O(1)` |
| 5. Bảng DP | `O(k)` | `O(k)` |

Đo thử với `n = 30` (các cách nhanh) và một số `k` cỡ vừa cho Cách 5.

In [24]:
random.seed(779)
cases = [(30, 1), (30, 2 ** 29), (30, 2 ** 28), (30, 2 ** 29 - 1)] + [(30, random.randint(1, 2 ** 29)) for _ in range(20000)]
ref = None
for name in ["halves", "parent", "popcount"]:
    t0 = time.perf_counter()
    res = [solutions[name](n, k) for n, k in cases]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:9} {len(cases)} truy vấn n=30: {dt:8.2f} ms")

for K in (10 ** 5, 10 ** 6, 3 * 10 ** 6):
    t0 = time.perf_counter()
    v = kth_table(30, K)
    dt = (time.perf_counter() - t0) * 1000
    assert v == kth_popcount(30, K)
    print(f"table  k={K:>9,}: {dt:8.1f} ms  (kết quả {v})")

halves    20004 truy vấn n=30:    63.01 ms
parent    20004 truy vấn n=30:    48.42 ms
popcount  20004 truy vấn n=30:     1.83 ms
table  k=  100,000:      6.0 ms  (kết quả 0)
table  k=1,000,000:     62.3 ms  (kết quả 0)


table  k=3,000,000:    189.8 ms  (kết quả 1)


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với nhau và với oracle, và test ngẫu nhiên có seed (đúng ràng buộc, `n` tới 30).

In [25]:
# 12.1 ví dụ đề
for n, k, expected in examples:
    for name, fn in solutions.items():
        assert fn(n, k) == expected, (name, n, k)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay theo các hàng ở mục 4)
rows = {1: "0", 2: "01", 3: "0110", 4: "01101001", 5: "0110100110010110"}
edge = []
for n, row in rows.items():
    for k in range(1, len(row) + 1):
        edge.append((n, k, int(row[k - 1])))
for n, k, expected in edge:
    for name, fn in solutions.items():
        assert fn(n, k) == expected, (name, n, k)
print("12.2 toàn bộ hàng 1..5 (viết tay):", len(edge), "ca: OK")

# 12.3 phản ví dụ: các cách sai
def wrong_k_not_minus_1(n, k):          # lệch chỉ số: dùng k thay vì k - 1
    return k.bit_count() & 1

def wrong_no_flip(n, k):                # nửa sau không đảo
    if n == 1:
        return 0
    half = 1 << (n - 2)
    if k <= half:
        return wrong_no_flip(n - 1, k)
    return wrong_no_flip(n - 1, k - half)

def wrong_half(n, k):                   # nửa = 2^(n-1) (độ dài cả hàng) thay vì 2^(n-2)
    if n == 1:
        return 0
    half = 1 << (n - 1)
    if k <= half:
        return wrong_half(n - 1, k)
    return wrong_half(n - 1, k - half) ^ 1

def wrong_parity_of_k(n, k):            # chỉ xét k chẵn/lẻ
    return (k - 1) & 1

wrongs = [
    ("lệch chỉ số (k thay vì k-1)", wrong_k_not_minus_1, 3, 3),
    ("quên đảo ở nửa sau", wrong_no_flip, 2, 2),
    ("nửa = 2^(n-1)", wrong_half, 3, 4),
    ("chỉ xét tính chẵn lẻ của k", wrong_parity_of_k, 4, 6),
]
for name, fn, n, k in wrongs:
    truth = kth_build(n, k)
    print(f"{name:30} n={n} k={k}: sai = {fn(n, k)}, đúng = {truth}")
    assert fn(n, k) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

# 12.4 ngẫu nhiên có seed với n tới 30: các cách nhanh khớp nhau và khớp bảng khi k vừa
rng = random.Random(2779)
for _ in range(3000):
    n = rng.randint(1, 30)
    k = rng.randint(1, 2 ** (n - 1))
    res = {name: solutions[name](n, k) for name in ("halves", "parent", "popcount", "popcount_bin", "sol_halves", "sol_parent", "sol_popcount", "final")}
    assert len(set(res.values())) == 1, (n, k, res)
    if k <= 20000:
        assert kth_table(n, k) == res["final"], (n, k)
print("12.4 ngẫu nhiên n tới 30: OK")

# 12.5 đối chiếu với oracle dựng hàng ở n = 21 (1 triệu ký tự) tại các k ngẫu nhiên và biên
row21 = build_row(21)
assert len(row21) == 2 ** 20
rng = random.Random(21)
ks = [1, 2, 2 ** 19, 2 ** 19 + 1, 2 ** 20] + [rng.randint(1, 2 ** 20) for _ in range(5000)]
for k in ks:
    assert sol.kthGrammar(21, k) == kth_halves(21, k) == kth_parent(21, k) == int(row21[k - 1]), k
print("12.5 đối chiếu hàng 21 (1.048.576 ký tự): OK")

# 12.6 tính chất: số 1 trong hàng n là 2^(n-2) (n >= 2); nửa sau là đảo nửa đầu: s[i] != s[i + len/2]
for n in range(2, 13):
    row = build_row(n)
    assert row.count("1") == 2 ** (n - 2)
    h = len(row) // 2
    assert all(row[i] != row[i + h] for i in range(h))
print("12.6 tính chất của hàng: OK")

12.1 ví dụ đề: OK
12.2 toàn bộ hàng 1..5 (viết tay): 31 ca: OK
lệch chỉ số (k thay vì k-1)    n=3 k=3: sai = 0, đúng = 1
quên đảo ở nửa sau             n=2 k=2: sai = 0, đúng = 1
nửa = 2^(n-1)                  n=3 k=4: sai = 1, đúng = 0
chỉ xét tính chẵn lẻ của k     n=4 k=6: sai = 1, đúng = 0
12.3 phản ví dụ: OK (các cách sai đều bị lộ)
12.4 ngẫu nhiên n tới 30: OK


12.5 đối chiếu hàng 21 (1.048.576 ký tự): OK
12.6 tính chất của hàng: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Lệch chỉ số: dùng `k` thay cho `k - 1`

Đề đánh số từ 1; công thức popcount dùng chỉ số từ 0.

In [26]:
for n, k in [(2, 2), (3, 3), (4, 6)]:
    print(f"n={n} k={k}: sai (popcount(k)) = {wrong_k_not_minus_1(n, k)} | đúng = {kth_build(n, k)}")
assert wrong_k_not_minus_1(3, 3) != kth_build(3, 3)

n=2 k=2: sai (popcount(k)) = 1 | đúng = 1
n=3 k=3: sai (popcount(k)) = 0 | đúng = 1
n=4 k=6: sai (popcount(k)) = 0 | đúng = 0


Vì sao sai: với `k = 3` chỉ số từ 0 là `2 = 0b10` (một bit 1 -> `1`), nhưng `popcount(3) = popcount(0b11) = 2 -> 0`. (Với vài giá trị như `k = 2` hai công thức tình cờ trùng nhau, nên test ít ví dụ rất dễ bỏ sót lỗi.) **Đúng:** luôn đổi sang chỉ số từ 0 trước khi dùng công thức bit: `(k - 1).bit_count() & 1`.

### Lỗi 2 — Quên đảo ở nửa sau

Nếu viết `return f(n - 1, k - half)` (không `^ 1`), mọi ký tự ở nửa sau sao chép sai.

In [27]:
print("n=2, k=2: sai =", wrong_no_flip(2, 2), "| đúng =", kth_build(2, 2))
print("n=3, k=3: sai =", wrong_no_flip(3, 3), "| đúng =", kth_build(3, 3))
assert wrong_no_flip(2, 2) == 0 and kth_build(2, 2) == 1

n=2, k=2: sai = 0 | đúng = 1
n=3, k=3: sai = 0 | đúng = 1


Vì sao sai: nửa sau của hàng là **đảo** của hàng trước. Không đảo thì hàng `n` luôn chỉ gồm `0` (sai hoàn toàn). **Đúng:** `... ^ 1` ở nhánh `k > half`.

### Lỗi 3 — Dùng nửa là `2^(n-1)` thay vì `2^(n-2)`

`2^(n-1)` là độ dài **cả hàng `n`**; nửa hàng `n` là `2^(n-2)` (chính là độ dài hàng `n - 1`).

In [28]:
print("n=3, k=4: sai =", wrong_half(3, 4), "| đúng =", kth_build(3, 4))
assert wrong_half(3, 4) != kth_build(3, 4)

n=3, k=4: sai = 1 | đúng = 0


Vì sao sai: với `n = 3` hàng có độ dài 4, nửa là 2; dùng 4 khiến mọi `k` đều bị coi là nằm ở nửa đầu và ký tự cuối bị tính sai. **Đúng:** `half = 1 << (n - 2)` (và đúng bài: `n = 1` đã xử lý riêng).

### Lỗi 4 — Dựng cả hàng cho `n` lớn

Với `n = 30`, hàng dài `2^29` ký tự: hết bộ nhớ và thời gian. Hãy xem thời gian và độ dài khi tăng `n` (đã đo ở mục 5).

In [29]:
for n in (20, 24, 28, 30):
    print(f"n={n}: hàng có {2 ** (n - 1):>13,} ký tự, nếu mỗi ký tự 1 byte thì ~ {2 ** (n - 1) / 2 ** 20:>8,.0f} MB")
print("trong khi đáp án chỉ cần", sol.kthGrammar(30, 123456789), "(một dòng bit_count)")

n=20: hàng có       524,288 ký tự, nếu mỗi ký tự 1 byte thì ~        0 MB
n=24: hàng có     8,388,608 ký tự, nếu mỗi ký tự 1 byte thì ~        8 MB
n=28: hàng có   134,217,728 ký tự, nếu mỗi ký tự 1 byte thì ~      128 MB
n=30: hàng có   536,870,912 ký tự, nếu mỗi ký tự 1 byte thì ~      512 MB
trong khi đáp án chỉ cần 1 (một dòng bit_count)


Vì sao sai: bài chỉ hỏi **một** ký tự; dựng toàn bộ là lãng phí lũy thừa. **Đúng:** khai thác cấu trúc (đệ quy theo nửa hoặc popcount).

### Lỗi 5 — Chỉ xét tính chẵn lẻ của `k`

Nghĩ "`k` chẵn thì `1`, lẻ thì `0`" vì hàng 2 là `01`. Nhưng ở các hàng sau, phép đảo tích lũy qua nhiều tầng.

In [30]:
for n, k in [(4, 6), (3, 3), (3, 4)]:
    print(f"n={n} k={k}: sai (k-1)&1 = {wrong_parity_of_k(n, k)} | đúng = {kth_build(n, k)}")
assert wrong_parity_of_k(4, 6) != kth_build(4, 6)

n=4 k=6: sai (k-1)&1 = 1 | đúng = 0
n=3 k=3: sai (k-1)&1 = 0 | đúng = 1
n=3 k=4: sai (k-1)&1 = 1 | đúng = 0


Vì sao sai: đảo xảy ra ở **mỗi** tầng cha-con nên đáp án phụ thuộc **số** lần đảo (parity của số bit 1), không chỉ tầng cuối. **Đúng:** đếm hết các bit của `k - 1`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** dãy sinh bằng quy tắc thay thế lặp, độ dài tăng theo lũy thừa, hỏi **một** phần tử -> tìm **tự tương tự** (nửa đầu/nửa sau) hoặc quan hệ **cha-con** rồi đi ngược.
- Dãy này là **dãy Thue-Morse**: phần tử thứ `i` (từ 0) bằng `popcount(i) mod 2`. Nó còn tương ứng với mã Gray khác một chút (mã Gray phản chiếu, còn đây đảo bit).
- Khi đệ quy mỗi lần chỉ gọi một nhánh (đuôi), thường chuyển được thành **vòng lặp** hoặc công thức bit.
- Lệch chỉ số 1/0 là lỗi số một: quy đổi `k -> k - 1` trước khi dùng bit.
- Hàng `n` dài `2^(n-1)`; kiểm tra nhanh `k <= 2^(n-1)`.
- `n` không ảnh hưởng đáp án ở Cách 3 và 4 (với `k` hợp lệ).
- Test nên có: `k = 1` (luôn `0`), `k = 2^(n-1)` (ký tự cuối: bằng `0` khi `n` lẻ, `1` khi `n` chẵn), các hàng nhỏ viết tay.

## 15. Biến thể

1. **Số ký tự `1` trong `m` ký tự đầu** của dãy: mỗi cặp `(2j, 2j + 1)` có đúng một `1`, nên trong `m` ký tự đầu có `m // 2` số `1` cộng thêm 1 nếu `m` lẻ và ký tự cuối (chỉ số `m - 1`) là `1`.
2. **Quy tắc ba ký hiệu:** `0 -> 012`, `1 -> 120`, `2 -> 201` (bài tập 5): ký tự là **tổng chữ số hệ 3** của `k - 1`, mod 3.
3. **Thay thế khác:** `0 -> 01`, `1 -> 00`? (không còn đối xứng; thường cần đệ quy với đếm).
4. Bài liên quan: Gray Code (89), Generate Parentheses theo cấu trúc đệ quy, Find K-th Bit in Nth Binary String (1545: cùng họ, có thêm đảo + đảo thứ tự).

Dưới đây là code cho biến thể 1, kiểm bằng oracle dựng dãy.

In [31]:
def tm(i: int) -> int:
    return i.bit_count() & 1

def count_ones_prefix(m: int) -> int:
    """Số ký tự 1 trong m ký tự đầu (chỉ số 0..m-1) của dãy Thue-Morse."""
    return m // 2 + (tm(m - 1) if m % 2 else 0)

seq = [tm(i) for i in range(2 ** 12)]
for m in range(0, 2 ** 12 + 1):
    assert count_ones_prefix(m) == sum(seq[:m]), m
assert seq[:16] == [int(c) for c in build_row(5)]
print("số 1 trong 6 ký tự đầu:", count_ones_prefix(6), "| 7 ký tự đầu:", count_ones_prefix(7))
print("Biến thể: OK")

số 1 trong 6 ký tự đầu: 3 | 7 ký tự đầu: 3
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng hàng | Mô phỏng đúng đề | `O(2^n)` | `O(2^n)` | Oracle, `n <= 20` |
| 2. Đệ quy theo nửa | Nửa đầu giữ, nửa sau đảo | `O(n)` | `O(n)` | Khi muốn bám cấu trúc đề; dễ giải thích |
| 3. Leo cha | `k -> (k+1)//2`, đếm `k` chẵn | `O(log k)` | `O(1)` | Không dùng `n`; dễ viết vòng lặp |
| 4. Popcount | `popcount(k-1) & 1` | `O(1)` | `O(1)` | **Cách nộp gọn nhất** |
| 5. Bảng Thue-Morse | `t[i] = t[i//2] ^ (i&1)` | `O(k)` | `O(k)` | Minh họa DP; `k` nhỏ |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(build_row(4))
print((5 - 1).bit_count() & 1)
print(Solution().kthGrammar(4, 5))
```

In [32]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Dùng cách leo cha cho `n = 5, k = 11`: ghi dãy các giá trị `k` qua từng bước (bắt đầu `11`), số lần đảo và đáp án.

In [33]:
# Bài 2: điền kết quả tính tay
hand_k_chain = None     # ví dụ [11, ..., 1]
hand_flips = None
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành đệ quy theo nửa.

In [34]:
def exercise_3_halves(n: int, k: int) -> int:
    # TODO 1: điều kiện dừng khi n == 1
    # TODO 2: half = độ dài nửa hàng n (dùng 1 << (n - 2))
    # TODO 3: nếu k <= half thì gọi lại với (n - 1, k)
    # TODO 4: ngược lại gọi (n - 1, k - half) và đảo bit bằng ^ 1
    return None


# assert exercise_3_halves(1, 1) == 0
# assert exercise_3_halves(2, 2) == 1
# assert exercise_3_halves(4, 6) == 0

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_parity(n, k)` đếm số bit 1 của `k - 1` **mà không dùng `bit_count` hay `bin().count`** (dùng vòng lặp `x & (x - 1)` hoặc dịch bit).

In [35]:
def exercise_4_parity(n: int, k: int) -> int:
    # TODO
    pass


# assert exercise_4_parity(2, 2) == 1
# assert exercise_4_parity(4, 6) == 0
# assert exercise_4_parity(30, 2 ** 29) == 1

### Bài 5 — Mini challenge: quy tắc ba ký hiệu (khó)

Bảng `n` hàng; hàng 1 là `0`; mỗi hàng sau thay `0 -> 012`, `1 -> 120`, `2 -> 201`. Viết `exercise_5_ternary(n, k)` trả ký tự thứ `k` (**1-indexed**) của hàng `n` (hàng `n` dài `3^(n-1)`).

```python
exercise_5_ternary(2, 1) -> 0     (hàng 2: 012)
exercise_5_ternary(2, 3) -> 2
exercise_5_ternary(3, 5) -> 2     (hàng 3: 012120201)
```
Gợi ý: ký tự con thứ `j` (0, 1, 2) của cha `c` là `(c + j) mod 3`; đi ngược từ `k - 1` theo hệ 3.

In [36]:
def exercise_5_ternary(n: int, k: int) -> int:
    # TODO
    pass


# assert exercise_5_ternary(2, 1) == 0
# assert exercise_5_ternary(2, 3) == 2
# assert exercise_5_ternary(3, 5) == 2

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: hàng 4 đã liệt kê ở mục 4.
- Mức 2: `5 - 1 = 4 = 0b100` có đúng một bit 1.
- Mức 3: `01101001`, `1`, `1` (ký tự thứ 5 của `01101001` là `1`).

**Bài 2**
- Mức 1: nếu `k` chẵn thì `flips += 1`; sau đó `k = (k + 1) // 2`.
- Mức 2: `11 -> 6 -> 3 -> 2 -> 1`.
- Mức 3: chẵn tại `6` và `2` (hai lần đảo): `flips = 2` -> đáp án `0`; chuỗi `[11, 6, 3, 2, 1]`.

**Bài 3**
- Mức 1: `if n == 1: return 0`.
- Mức 2: `half = 1 << (n - 2)`.
- Mức 3: `if k <= half: return f(n - 1, k)`; `return f(n - 1, k - half) ^ 1`.

**Bài 4**
- Mức 1: `x = k - 1`, đếm bit.
- Mức 2: `while x: cnt += x & 1; x >>= 1` hoặc `x &= x - 1`.
- Mức 3: `return cnt & 1`.

**Bài 5**
- Mức 1: ký tự ở chỉ số 0 `i` bằng `(ký tự cha + (i mod 3)) mod 3` với cha chỉ số `i // 3`.
- Mức 2: lặp: `i = k - 1`; cộng dần `i % 3`, rồi `i //= 3`.
- Mức 3: kết quả = tổng chữ số hệ 3 của `k - 1`, lấy mod 3.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
print(build_row(4))
print((5 - 1).bit_count() & 1)
print(Solution().kthGrammar(4, 5))

assert build_row(4) == "01101001"
assert (5 - 1).bit_count() & 1 == 1
assert Solution().kthGrammar(4, 5) == 1 == int(build_row(4)[4])

01101001
1
1


### Đáp án Bài 2

Chuỗi `k`: `11 -> 6 -> 3 -> 2 -> 1`. Chẵn tại `6` và `2` (2 lần đảo) -> `0`. Kiểm tra hàng 5: ký tự thứ 11 là `0`.

In [38]:
hand_k_chain_solution = [11, 6, 3, 2, 1]
hand_flips_solution = 2
hand_answer_solution = 0

chain, k, flips = [11], 11, 0
while k > 1:
    if k % 2 == 0:
        flips += 1
    k = (k + 1) // 2
    chain.append(k)
assert chain == hand_k_chain_solution and flips == hand_flips_solution
assert flips & 1 == hand_answer_solution == int(build_row(5)[10])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_halves_solution(n: int, k: int) -> int:
    if n == 1:
        return 0
    half = 1 << (n - 2)
    if k <= half:
        return exercise_3_halves_solution(n - 1, k)
    return exercise_3_halves_solution(n - 1, k - half) ^ 1

assert exercise_3_halves_solution(1, 1) == 0
assert exercise_3_halves_solution(2, 2) == 1
assert exercise_3_halves_solution(4, 6) == 0
for n in range(1, 11):
    row = build_row(n)
    for k in range(1, len(row) + 1):
        assert exercise_3_halves_solution(n, k) == int(row[k - 1]), (n, k)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [40]:
def exercise_4_parity_solution(n: int, k: int) -> int:
    x = k - 1
    cnt = 0
    while x:
        x &= x - 1            # xóa bit 1 thấp nhất
        cnt += 1
    return cnt & 1

assert exercise_4_parity_solution(2, 2) == 1
assert exercise_4_parity_solution(4, 6) == 0
assert exercise_4_parity_solution(30, 2 ** 29) == 1
rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 30)
    k = rng.randint(1, 2 ** (n - 1))
    assert exercise_4_parity_solution(n, k) == kth_popcount(n, k), (n, k)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_ternary_solution(n: int, k: int) -> int:
    i = k - 1
    total = 0
    while i:
        total += i % 3
        i //= 3
    return total % 3

def ternary_row(n: int) -> str:
    row = "0"
    mapping = {"0": "012", "1": "120", "2": "201"}
    for _ in range(n - 1):
        row = "".join(mapping[c] for c in row)
    return row

assert exercise_5_ternary_solution(2, 1) == 0
assert exercise_5_ternary_solution(2, 3) == 2
assert exercise_5_ternary_solution(3, 5) == 2
assert ternary_row(3) == "012120201"
for n in range(1, 8):
    row = ternary_row(n)
    assert len(row) == 3 ** (n - 1)
    for k in range(1, len(row) + 1):
        assert exercise_5_ternary_solution(n, k) == int(row[k - 1]), (n, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (một dòng popcount / hoặc đệ quy theo nửa)

```python
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        return (k - 1).bit_count() & 1          # Thue-Morse: số bit 1 của chỉ số (từ 0)
```

Hoặc, bám cấu trúc đề:

```python
class Solution:
    def kthGrammar(self, n: int, k: int) -> int:
        if n == 1:
            return 0
        half = 1 << (n - 2)                      # độ dài hàng n - 1 = nửa hàng n
        if k <= half:
            return self.kthGrammar(n - 1, k)
        return self.kthGrammar(n - 1, k - half) ^ 1   # nửa sau = đảo
```

Công thức nhớ nhanh: hàng `n` = hàng `n-1` + đảo(hàng `n-1`); ký tự thứ `k` = `popcount(k - 1) mod 2`.

### Checklist trước khi submit

- [ ] Đã đổi sang chỉ số từ 0 (`k - 1`) trước khi dùng công thức bit?
- [ ] Ở nửa sau có `^ 1` (đảo)?
- [ ] `half = 1 << (n - 2)` (không phải `n - 1`)?
- [ ] Điều kiện dừng `n == 1` trả `0`?
- [ ] Không dựng cả hàng (với `n = 30` là `2^29` ký tự)?
- [ ] Đã thử `k = 1`, `k = 2^(n-1)`, `n = 1`, `n = 30`?
- [ ] Tên method đúng `kthGrammar(self, n, k)` và trả `int`?